# Recipe Recommender — Exploration

Dataset exploration, elbow-method plot, qualitative result inspection.

In [2]:
import sys
sys.path.insert(0, '..')

from src.recommender import RecipeRecommender, Filters
from src.kmeans import elbow_analysis, choose_k_elbow

engine = RecipeRecommender.build_or_load(use_semantic_normalizer=False)
print(f'Recipes: {len(engine.recipes)}  Vocab: {len(engine.vocabulary)}')

[build] Loading recipes from HuggingFace (this may take a few minutes)...


README.md: 0.00B [00:00, ?B/s]

d:\AppGallery\Work\anaconda3\envs\recipe_recommender\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\XSIMBA\.cache\huggingface\hub\datasets--datahiveai--recipes-with-nutrition. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


recipes-with-nutrition.csv:   0%|          | 0.00/450M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

[build] Loaded 39447 recipes
[build] Building ingredient vocabulary...
[build] Vocabulary size: 3511
[build] Fitting TF-IDF...
[build] TF-IDF matrix: (39447, 3511), nnz=339966
[build] Building 2-D projection for clustering and visualization...
[build] Choosing k via elbow analysis...
[build] Selected k=9. Inertias: {5: 155.71634630680228, 7: 112.58325906517206, 9: 87.47674513873395, 11: 72.45797909746851, 13: 61.62362618244191, 15: 53.68672707865491, 17: 49.00598405896861}
[build] Fitting K-Means with k=9...


[nltk_data] Error loading wordnet: Remote end closed connection
[nltk_data]     without response


[build] Saving engine cache...
[build] Done.
Recipes: 39447  Vocab: 3511


## Elbow analysis

In [3]:
import plotly.graph_objects as go

ks = list(range(4, 21, 2))
inertias = elbow_analysis(engine.projection_2d, ks)
chosen = choose_k_elbow(inertias)

fig = go.Figure()
fig.add_trace(go.Scatter(x=ks, y=[inertias[k] for k in ks], mode='lines+markers'))
fig.add_trace(go.Scatter(x=[chosen], y=[inertias[chosen]], mode='markers',
                         marker=dict(size=14, color='red'), name=f'chosen k={chosen}'))
fig.update_layout(title='Elbow analysis', xaxis_title='k', yaxis_title='inertia')
fig.show()

## Qualitative inspection

In [4]:
queries = [
    ['chicken', 'tomato', 'onion', 'garlic'],
    ['rice', 'soy sauce', 'ginger', 'tofu'],
    ['洋葱', '番茄', 'chiken'],
    ['flour', 'sugar', 'butter', 'egg', 'cocoa'],
]
for q in queries:
    print('\nQuery:', q)
    for r in engine.recommend(q, top_k=5):
        print(f'  {r.score:.3f}  {r.recipe.name[:60]}  missing={r.missing[:3]}')


Query: ['chicken', 'tomato', 'onion', 'garlic']
  0.604  recipe-13407  missing=[]
  0.586  recipe-1396  missing=['kosher salt']
  0.555  recipe-8984  missing=[]
  0.489  recipe-874  missing=['lemon juice']
  0.484  recipe-27019  missing=['fresh basil']

Query: ['rice', 'soy sauce', 'ginger', 'tofu']
  0.662  recipe-1572  missing=[]
  0.606  recipe-34100  missing=[]
  0.604  recipe-3856  missing=[]
  0.565  recipe-7153  missing=[]
  0.414  recipe-9046  missing=['kosher salt']

Query: ['洋葱', '番茄', 'chiken']
  0.628  recipe-13407  missing=[]
  0.623  recipe-1396  missing=['kosher salt']
  0.577  recipe-8984  missing=[]
  0.507  recipe-26664  missing=['sage']
  0.470  recipe-9593  missing=['kosher salt']

Query: ['flour', 'sugar', 'butter', 'egg', 'cocoa']
  0.751  recipe-16644  missing=[]
  0.724  recipe-31915  missing=[]
  0.717  recipe-3862  missing=[]
  0.710  recipe-102  missing=[]
  0.667  recipe-22631  missing=[]
